In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
import pyarrow.parquet as pq
import os
import time
import joblib

# --- 1. Data Pipeline for Reduced Dataset ---

def create_gan_dataset(filepath, batch_size):
    """
    Creates a tf.data.Dataset to stream the reduced-dimension data
    from the PCA-processed Parquet file.
    """
    if not os.path.exists(filepath):
        print(f"Error: The file at {filepath} was not found.")
        return None, None, None, None

    try:
        parquet_file = pq.ParquetFile(filepath)
        column_names = parquet_file.schema.names
        n_features = len(column_names)
        print(f"Loading reduced dataset with {n_features} features.")

        # --- Fit Scaler ---
        # The PCA data is standardized, but WGANs work best with data scaled to [-1, 1].
        print("Fitting a new scaler for the GAN training range [-1, 1]...")
        sample_df = parquet_file.read_row_group(0).to_pandas()
        scaler = MinMaxScaler(feature_range=(-1, 1))
        scaler.fit(sample_df)
        print("Scaler fitted.")

        def data_generator():
            """A generator that yields rows from the parquet file."""
            for i in range(parquet_file.num_row_groups):
                df = parquet_file.read_row_group(i).to_pandas()
                scaled_data = scaler.transform(df)
                for row in scaled_data:
                    yield row

        dataset = tf.data.Dataset.from_generator(
            data_generator,
            output_signature=tf.TensorSpec(shape=(n_features,), dtype=tf.float32)
        )

        # Prepare the dataset for efficient training
        dataset = dataset.shuffle(buffer_size=1000).batch(batch_size, drop_remainder=True).repeat().prefetch(buffer_size=tf.data.AUTOTUNE)
        
        num_rows = parquet_file.metadata.num_rows
        steps_per_epoch = num_rows // batch_size

        return dataset, scaler, column_names, steps_per_epoch

    except Exception as e:
        print(f"An error occurred while creating the data pipeline: {e}")
        return None, None, None, None


# --- 2. Build the WGAN-GP Models ---

def build_critic(n_inputs):
    """Builds the Critic model."""
    model = Sequential(name="Critic")
    model.add(Dense(512, input_dim=n_inputs))
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions
    model.add(Dropout(0.3))
    
    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions
    model.add(Dropout(0.3))

    model.add(Dense(128))
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions
    model.add(Dropout(0.3))

    model.add(Dense(1)) # No activation function for WGAN critic
    return model

def build_generator(latent_dim, n_outputs):
    """Builds the Generator model."""
    model = Sequential(name="Generator")
    model.add(Dense(128, input_dim=latent_dim))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions

    model.add(Dense(256))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions
    
    model.add(Dense(512))
    model.add(BatchNormalization())
    model.add(LeakyReLU(alpha=0.2)) # FIX: Use 'alpha' for older TF versions

    model.add(Dense(n_outputs, activation='tanh')) # tanh for output between -1 and 1
    return model

# --- 3. WGAN-GP Training Logic ---

class WGAN(tf.keras.Model):
    def __init__(self, generator, critic, latent_dim, critic_extra_steps=5, gp_weight=10.0):
        super(WGAN, self).__init__()
        self.generator = generator
        self.critic = critic
        self.latent_dim = latent_dim
        self.critic_extra_steps = critic_extra_steps
        self.gp_weight = gp_weight

    def compile(self, g_optimizer, c_optimizer):
        super(WGAN, self).compile()
        self.g_optimizer = g_optimizer
        self.c_optimizer = c_optimizer

    def gradient_penalty(self, batch_size, real_data, fake_data):
        """Calculates the gradient penalty."""
        alpha = tf.random.normal([batch_size, 1], 0.0, 1.0)
        diff = fake_data - real_data
        interpolated = real_data + alpha * diff

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic(interpolated, training=True)

        grads = gp_tape.gradient(pred, [interpolated])[0]
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1]))
        gp = tf.reduce_mean((norm - 1.0) ** 2)
        return gp

    def train_step(self, real_data):
        batch_size = tf.shape(real_data)[0]

        # Train the critic more often than the generator
        for i in range(self.critic_extra_steps):
            with tf.GradientTape() as tape:
                random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
                fake_data = self.generator(random_latent_vectors, training=True)
                
                fake_logits = self.critic(fake_data, training=True)
                real_logits = self.critic(real_data, training=True)

                c_cost = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits)
                gp = self.gradient_penalty(batch_size, real_data, fake_data)
                c_loss = c_cost + gp * self.gp_weight
            
            c_gradient = tape.gradient(c_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(zip(c_gradient, self.critic.trainable_variables))

        # Train the generator
        with tf.GradientTape() as tape:
            random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
            generated_data = self.generator(random_latent_vectors, training=True)
            gen_data_logits = self.critic(generated_data, training=True)
            g_loss = -tf.reduce_mean(gen_data_logits)
        
        gen_gradient = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(gen_gradient, self.generator.trainable_variables))
        
        return {"c_loss": c_loss, "g_loss": g_loss, "wasserstein_dist": -c_cost}

def generate_and_save_data(generator, scaler, latent_dim, column_names, n_samples, filename):
    """Generates synthetic data and saves it to a CSV file."""
    latent_points = np.random.randn(n_samples, latent_dim)
    synthetic_data = generator.predict(latent_points)
    synthetic_data_rescaled = scaler.inverse_transform(synthetic_data)
    df_synthetic = pd.DataFrame(synthetic_data_rescaled, columns=column_names)
    df_synthetic.to_csv(filename, index=False)
    print(f"\nGenerated and saved {n_samples} samples to {filename}\n")

# --- 4. Main Execution ---

if __name__ == "__main__":
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"Found {len(gpus)} GPUs. TensorFlow will use the GPU.")
        except RuntimeError as e:
            print(e)
    else:
        print("No GPU found. This will be very slow.")

    # --- Configuration ---
    FILEPATH = 'pivoted_data_reduced_pca.parquet' # Use the new reduced file
    LATENT_DIM = 128
    EPOCHS = 100
    BATCH_SIZE = 256 # Can use a larger batch size now

    # --- Create Data Pipeline ---
    dataset, scaler, column_names, steps_per_epoch = create_gan_dataset(FILEPATH, BATCH_SIZE)

    if dataset is not None:
        N_FEATURES = len(column_names)

        # --- Build and Compile WGAN-GP ---
        generator = build_generator(LATENT_DIM, N_FEATURES)
        critic = build_critic(N_FEATURES)
        
        wgan = WGAN(generator=generator, critic=critic, latent_dim=LATENT_DIM)

        g_optimizer = Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)
        c_optimizer = Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9)
        wgan.compile(g_optimizer=g_optimizer, c_optimizer=c_optimizer)

        # --- Train the Model ---
        print("Starting WGAN-GP training...")
        class GANMonitor(tf.keras.callbacks.Callback):
            def on_epoch_end(self, epoch, logs=None):
                print(f"Epoch {epoch+1}, Critic Loss={logs['c_loss']:.4f}, Gen Loss={logs['g_loss']:.4f}, Wasserstein Dist={logs['wasserstein_dist']:.4f}")
                if (epoch + 1) % 20 == 0:
                    generate_and_save_data(self.model.generator, scaler, LATENT_DIM, column_names, 1000, f"synthetic_data_epoch_{epoch+1}.csv")

        wgan.fit(dataset, epochs=EPOCHS, steps_per_epoch=steps_per_epoch, callbacks=[GANMonitor()])
        
        print("Training complete.")

        # --- Generate Final Dataset ---
        print("Generating final synthetic dataset...")
        generate_and_save_data(generator, scaler, LATENT_DIM, column_names, 40000, "final_synthetic_dataset_wgan.csv")
        print("Final dataset saved.")



2025-10-14 19:03:44.137171: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-14 19:03:46.391372: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-14 19:03:46.393947: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-14 19:03:52.132552: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


No GPU found. This will be very slow.


2025-10-14 19:03:58.213286: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Loading reduced dataset with 200 features.
Fitting a new scaler for the GAN training range [-1, 1]...
Scaler fitted.
Starting WGAN-GP training...
Epoch 1/100
153/153 [==============================] - 19s 61ms/step - c_loss: -3.0390 - g_loss: 3.7420 - wasserstein_dist: 3.5609
Epoch 2/100
153/153 [==============================] - 8s 52ms/step - c_loss: -1.7206 - g_loss: 2.5374 - wasserstein_dist: 1.9113
Epoch 3/100
153/153 [==============================] - 9s 56ms/step - c_loss: -0.8800 - g_loss: 1.2744 - wasserstein_dist: 1.0048
Epoch 4/100
153/153 [==============================] - 10s 62ms/step - c_loss: -0.2676 - g_loss: 0.5823 - wasserstein_dist: 0.3694
Epoch 5/100
153/153 [==============================] - 10s 66ms/step - c_loss: -0.1226 - g_loss: 0.3371 - wasserstein_dist: 0.2191
Epoch 6/100
153/153 [==============================] - 10s 63ms/step - c_loss: -0.1644 - g_loss: 0.3120 - wasserstein_dist: 0.2575
Epoch 7/100
153/153 [==============================] - 9s 56ms/step - 